# Chapter 5 — Gaia DR4 epoch astrometry: refitting stars and orbits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/gaia-nss-tutorial/blob/main/notebooks/05_dr4_epoch_astrometry.ipynb)

**Goal.** Gaia DR3 published, for each star, the *result* of fitting a model to its measurements. Gaia DR4, scheduled
for 2 December 2026 (https://www.cosmos.esa.int/web/gaia/data-release-4), will also publish the measurements
themselves: the **epoch astrometry**, one along-scan position per CCD crossing, over 66 months. In June 2026 ESA
pre-released these data for 12 sources (https://www.cosmos.esa.int/web/gaia/dr4-prerelease): three quasars, six stars
without known companions, and three binaries — Gaia BH3, a 33 $M_\odot$ black hole orbiting a metal-poor giant
(Gaia Collaboration, Panuzzo et al. 2024, https://ui.adsabs.harvard.edu/abs/2024A&A...686L...2G), the planet host
Gaia-4 and the 84-day binary HD 114762. We refit all of them ourselves.

**You will learn**
1. what a DR4 epoch-astrometry record contains, and how to turn it into times, scan angles and along-scan positions;
2. to reproduce the DR4 single-star solution exactly (six parameters, a colour prior, the AGIS excess noise), and how
   the excess noise, not the $\chi^2$, carries the binarity signal of that solution;
3. how much DR4 improves on DR3, and why the ten CCD measurements of one transit are not ten independent epochs;
4. to fit a photocentre orbit, linear in the Thiele–Innes constants for fixed $(P, e, T_p)$, with uncertainties from
   resampling transits, and to turn it into Campbell elements, a mass function and a mass;
5. why DR3, with half the time span, could not find Gaia BH3, and what DR4 adds for the two other binaries.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, brentq     # orbit refinement; root of the mass-function equation

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts, attached to a GitHub release of the tutorial. MANIFEST.json lists, for each file,
# its origin (SQL query or URL) and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.2/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 5.1 What is in an epoch-astrometry record

The file `dr4_prerelease_epoch_astrometry` is the pre-release table with one row per CCD observation (the original
has one row per transit, with arrays over the CCDs). Each transit has 10 entries: the sky mapper (SM) and the nine
astrometric-field CCDs AF1–AF9. The columns we use (definitions from the DR4 pre-release data model):

- `obs_time_tcb` [ns]: observation time, TCB **at Gaia**, counted from JD 2455197.5 (2010 January 1, 0 h);
  `obs_time_bary_corr` [ns]: the light-time correction to the solar-system barycentre, TCB(barycentric) $-$ TCB(at
  Gaia), for this source, computed once per transit (at CCD AF4);
- `scan_pos_angle` $\theta$ [deg]: position angle of the scan direction, from north through east;
- `centroid_pos_al` $w$ [mas]: the measured along-scan position, relative to the reference point (`ra0`, `dec0`), and
  `centroid_pos_error_al` $\sigma$ [mas] its uncertainty;
- `parallax_factor_al` $f_{\rm AL}$: the along-scan displacement produced by a parallax of 1 mas;
- `colour_factor_al`: the along-scan displacement per unit error in the source colour (Section 5.2);
- `agis_source_excess_noise` $\epsilon$ [mas]: the excess noise of this source in the DR4 astrometric solution (AGIS),
  one value per source;
- `used_by_agis_al`: whether the observation was used by the DR4 astrometric solution. Rejected observations are
  not bad data — they are outliers, or CCDs the solution does not use — but the DR4 fit ignores them, and so do we.

The model of a single star is that of Chapter 1 (and of ESA's `gaiasupdate` package, https://github.com/esa/gaia-supdate),

$$ w = \Delta\alpha_*\sin\theta + \Delta\delta\cos\theta + \varpi f_{\rm AL} + (\mu_{\alpha*}\sin\theta + \mu_\delta\cos\theta)\,t , $$

with $t$ the barycentric time in years from the DR4 reference epoch J2017.5 (JD 2457936.875).

In [ ]:
raw = load('dr4_prerelease_epoch_astrometry')
JD_REF = 2457936.875                             # J2017.5, the DR4 reference epoch
# barycentric time in years from J2017.5: origin JD 2455197.5, plus (TCB at Gaia + barycentric correction) in ns,
# 86400e9 ns per day
raw['t'] = (2455197.5 + (raw.obs_time_tcb + raw.obs_time_bary_corr.astype(float)) / 86400e9 - JD_REF) / 365.25
ep = raw[raw.used_by_agis_al].copy()
NAMES = {2237987199365376: 'quasar', 60730287810150016: 'quasar', 10973744521070720: 'quasar',
         4181040337841125632: 'HD 183633', 3926186255616949504: 'parallax', 435469040545191680: 'parallax',
         1663617687609809280: 'LSPM J1324+6124', 2309425390592896: 'magnitude', 20694084440761600: 'magnitude',
         3937211745905473024: 'HD 114762', 1457486023639239296: 'Gaia-4', 4318465066420528000: 'Gaia BH3'}
BINARIES = ['HD 114762', 'Gaia-4', 'Gaia BH3']
BH3 = 4318465066420528000
print(f'{len(raw):,} CCD entries in {raw.transit_id.nunique()} transits; {len(ep):,} used by the DR4 solution')
print('fraction used, by CCD (0 = SM, 1-9 = AF1-AF9):', raw.groupby('ccd').used_by_agis_al.mean().round(2).to_list())
# every used observation has a time and a barycentric correction, and no sky-mapper observation is used
assert ep.t.notna().all() and (ep.ccd > 0).all()

rb = raw[raw.source_id == BH3]
bh = ep[ep.source_id == BH3]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(2017.5 + rb.t[~rb.used_by_agis_al], rb.centroid_pos_al[~rb.used_by_agis_al], 'x', color='0.6', ms=4,
           label='not used by the DR4 solution')
ax[0].plot(2017.5 + bh.t, bh.centroid_pos_al, '.', ms=3, label='used')
ax[0].set(xlabel='time [yr]', ylabel='along-scan position $w$ [mas]', title='Gaia BH3: all CCD observations')
ax[0].legend(fontsize=8)
k = rb.transit_id == bh.transit_id.iloc[100]                # one transit
u = k & rb.used_by_agis_al
ax[1].errorbar(rb.ccd[u], rb.centroid_pos_al[u], rb.centroid_pos_error_al[u], fmt='o')
ax[1].plot(rb.ccd[k & ~rb.used_by_agis_al], rb.centroid_pos_al[k & ~rb.used_by_agis_al], 'x', color='0.6', ms=8)
ax[1].set_xticks(range(10), ['SM'] + [f'AF{i}' for i in range(1, 10)])
ax[1].set(xlabel='CCD', ylabel='$w$ [mas]',
          title=f'one transit ({2017.5 + rb.t[k].iloc[0]:.3f}, scan angle {rb.scan_pos_angle[k].iloc[0]:.0f}°)')
plt.tight_layout(); plt.show()

**Figure 5.1.** Left: along-scan positions $w$ of Gaia BH3 against time; dots: CCD observations used by the DR4
solution, grey crosses: the others. Right: the CCD observations of a single transit, with their uncertainties.

The positions jump by hundreds of mas between transits (left): $w$ is a projection, and each transit has its own scan
angle, so the large proper motion of this halo star ($\approx 150$ mas/yr) appears with a different sign and size each
time. Within one transit (right) the CCD measurements agree to their uncertainties, ~0.1 mas for this $G = 11.2$ star.
About 74% of all CCD entries are used. The sky-mapper CCD never is; of the 1605 rejected AF observations, 89% belong
to transits rejected as a whole, among them the 97 transits that have no barycentric correction at all.

## 5.2 The DR4 single-star solution, reproduced

The DR4 astrometric solution fits each source with **six** parameters, not five (Lindegren et al. 2021,
https://ui.adsabs.harvard.edu/abs/2021A&A...649A...2L, describe the same scheme for DR3). The image position on the
CCD depends slightly on the colour of the source; the solution uses the colour measured by the photometry, and the
sixth parameter, multiplying `colour_factor_al`, absorbs an error in it. For the DR4-like fit this parameter carries a
Gaussian prior of width $0.085\times10^{-3}\,{\rm nm}^{-1}$, so it only moves where the data demand it, and the
column enters multiplied by $-10^3$ (a sign and unit convention). Each observation is weighted by
$1/(\sigma^2 + \epsilon^2)$: the CCD uncertainty plus the source's excess noise from the DR4 solution.

This is exactly what `gaiasupdate` 0.1.2 does (`GaiaEpochAstrometryArchive.supdate`), and it is a weighted linear
least-squares problem with a prior on one parameter: we add $1/0.085\times10^{-3}$ squared to that diagonal element
of the normal matrix. Below we check our 20 lines against the values `gaiasupdate` returns for the same data
(computed with the package in the companion project https://github.com/vasilybelokurov/gaia-dr4-explorer, file
`src/gaia_dr4_explorer/resources/prerelease_reference.csv`).

In [ ]:
COLOUR_PRIOR = 0.085e-3                          # width of the Gaussian prior on the colour parameter [1/nm]


def design(s, orbit=None):
    """Design matrix of the DR4 source model for the CCD observations in s.

    Columns: Delta alpha*, Delta delta, parallax, pm_ra*, pm_dec, colour; with orbit = (P [yr], e, tp [yr from J2017.5])
    four more, multiplying the Thiele-Innes constants A, F, B, G (Delta delta = A X + F Y, Delta alpha* = B X + G Y)."""
    th = np.radians(s.scan_pos_angle.to_numpy())
    t = s.t.to_numpy()
    sn, cs = np.sin(th), np.cos(th)
    cols = [sn, cs, s.parallax_factor_al.to_numpy(float), t * sn, t * cs, -1e3 * s.colour_factor_al.to_numpy()]
    if orbit is not None:
        X, Y = kepler_xy(t, *orbit)
        cols += [X * cs, Y * cs, X * sn, Y * sn]
    return np.column_stack(cols)


def kepler_xy(t, P, e, tp):
    """Elliptical rectangular coordinates X = cos E - e, Y = sqrt(1 - e^2) sin E (t, P, tp in the same units)."""
    M = 2 * np.pi * (t - tp) / P
    E = M.copy()
    for _ in range(60):                          # Newton iterations for E - e sin E = M
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return np.cos(E) - e, np.sqrt(1 - e**2) * np.sin(E)


def fit(s, orbit=None, excess=True, colour=True):
    """Weighted least squares with the colour prior. Weights 1/(sigma^2 + eps^2), or 1/sigma^2 with excess=False.

    Returns parameters, their covariance, chi^2 against the weights used, chi^2 against sigma^2 alone, and nu."""
    M = design(s, orbit)
    if not colour:
        M = np.delete(M, 5, axis=1)
    w, sig2 = s.centroid_pos_al.to_numpy(), s.centroid_pos_error_al.to_numpy()**2
    var = sig2 + (s.agis_source_excess_noise.iloc[0]**2 if excess else 0)
    N, b = M.T @ (M / var[:, None]), M.T @ (w / var)                       # normal equations
    if colour:
        N[5, 5] += 1 / COLOUR_PRIOR**2                                     # Gaussian prior on the colour parameter
    p = np.linalg.solve(N, b)
    r = w - M @ p
    return p, np.linalg.inv(N), np.sum(r**2 / var), np.sum(r**2 / sig2), len(w) - M.shape[1]


# gaiasupdate 0.1.2 on the same data: parallax [mas] and chi^2 (against the total variance)
GSU = {2237987199365376: (-0.002339354, 748.39896), 2309425390592896: (0.926210546, 586.29543),
       10973744521070720: (0.002650360, 530.35179), 20694084440761600: (4.942145281, 586.02051),
       60730287810150016: (0.003210282, 606.33470), 435469040545191680: (0.997733905, 695.14294),
       1457486023639239296: (13.624072430, 901.42193), 1663617687609809280: (10.106683470, 836.99099),
       3926186255616949504: (0.999329310, 471.64158), 3937211745905473024: (25.631658773, 583.32262),
       4181040337841125632: (1.001305014, 599.38843), 4318465066420528000: (1.960750539, 555.74552)}
rows = []
for sid, s in ep.groupby('source_id'):
    p, C, chi2, chi2_sig, nu = fit(s)
    p5, C5, *_ = fit(s, colour=False)
    rows.append(dict(source_id=sid, name=NAMES[sid], G=s.g_mag.iloc[0], n_obs=len(s), eps=s.agis_source_excess_noise.iloc[0],
                     parallax=p[2], parallax_error=np.sqrt(C[2, 2]), pmra=p[3], pmdec=p[4],
                     pmra_error=np.sqrt(C[3, 3]), pmdec_error=np.sqrt(C[4, 4]),
                     uwe=np.sqrt(chi2 / nu), uwe_sigma_only=np.sqrt(chi2_sig / nu),
                     colour_shift=(p[2] - p5[2]) / np.sqrt(C[2, 2]),
                     d_parallax_gsu=p[2] - GSU[sid][0], d_chi2_gsu=chi2 - GSU[sid][1]))
tab = pd.DataFrame(rows).set_index('source_id')
print(tab[['name', 'G', 'n_obs', 'eps', 'parallax', 'parallax_error', 'd_parallax_gsu', 'd_chi2_gsu']].to_string())
# reproducibility: our fit equals gaiasupdate's for all 12 sources (the reference values are given to 1e-9 mas, 1e-5)
assert (tab.d_parallax_gsu.abs() < 1e-8).all() and (tab.d_chi2_gsu.abs() < 1e-4).all()

Our parallaxes agree with `gaiasupdate` to the precision of the stored reference values, and so do the $\chi^2$. The
DR4 source model is nothing more than this: six linear parameters, one prior, and fixed weights.

Two statistics come out of the fit, and they must not be confused. The **UWE against the total variance**
$\sigma^2+\epsilon^2$ is close to 1 for every source by construction: $\epsilon$ was chosen by the DR4 solution to
make it so. The **UWE against the CCD uncertainties alone** measures how badly the single-star model fits.
(`gaiasupdate` reports the second; neither is the catalogue's `astrometric_gof_al`.) For a binary, the binarity signal
of the single-star solution is therefore the excess noise $\epsilon$ itself — the DR4 counterpart of the DR3
`astrometric_excess_noise` of Chapter 2.

In [ ]:
order = tab.sort_values('uwe_sigma_only')
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(len(order))
ax[0].semilogy(x, order.uwe_sigma_only, 'o', label='against $\\sigma^2$ (CCD uncertainties)')
ax[0].semilogy(x, order.uwe, 's', mfc='none', label='against $\\sigma^2 + \\epsilon^2$ (as weighted by DR4)')
ax[0].axhline(1.4, color='r', ls='--', lw=1)
ax[0].set_xticks(x, order.name, rotation=45, ha='right'); ax[0].set_ylabel('UWE of the single-star fit'); ax[0].legend(fontsize=8)
ax[1].bar(x, order.colour_shift)
ax[1].set_xticks(x, order.name, rotation=45, ha='right')
ax[1].set_ylabel('parallax: (6 parameters $-$ 5 parameters) / $\\sigma_\\varpi$')
plt.tight_layout(); plt.show()
single = tab[~tab.name.isin(BINARIES)]
for n, r in order.iterrows():
    print(f'{r["name"]:16s} G {r.G:5.2f}  eps {r.eps:6.3f} mas  UWE(sigma) {r.uwe_sigma_only:7.2f}  UWE(total) {r.uwe:5.3f}')
# calibration: the single-star model fits the quasars and stars without known companions to their CCD
# uncertainties, and the quasars' parallaxes are zero within 3 formal errors
qso = tab[tab.name == 'quasar']
assert single.uwe_sigma_only.between(0.9, 1.15).all() and (np.abs(qso.parallax / qso.parallax_error) < 3).all()
# the total variance makes every fit, binaries included, statistically acceptable
assert tab.uwe.between(0.9, 1.1).all()

**Figure 5.2.** Left: UWE $=\sqrt{\chi^2/\nu}$ of the DR4-like single-star fit of the 12 pre-release sources, in
increasing order, computed against the CCD uncertainties (dots) and against the total variance used as weights
(squares); dashed: 1.4. Right: change of the parallax when the colour parameter is added, in units of its uncertainty.

Against the CCD uncertainties alone, the quasars and the six stars without known companions have UWE 0.98–1.04: the
published uncertainties describe their scatter, and the model is complete. The three binaries stand out: Gaia-4
(UWE 1.9), HD 114762 (10) and Gaia BH3 (82). Against the total variance all twelve sit at $1.00$–$1.05$, because the
DR4 solution assigned each binary an excess noise $\epsilon$ (0.12, 1.23 and 6.58 mas) that absorbs its orbit; the
faint quasar with $\epsilon = 0.62$ mas shows that a non-zero $\epsilon$ is not by itself proof of a companion.

The colour parameter changes the parallaxes by less than $0.25\sigma$ (right): the largest shifts are for two of the
faint sources ($G = 19$–20) and for LSPM J1324+6124, an M dwarf; for most sources they are below $0.05\sigma$. Small,
but needed to reproduce DR4.

## 5.3 What doubling the baseline buys

For uniformly spread observations of fixed precision, the uncertainty of a position or parallax scales as
$N^{-1/2} \propto T^{-1/2}$ and that of a proper motion as $T^{-3/2}$ (the lever arm grows with $T$). From DR3
(34 months) to DR4 (66 months) the expected ratios are therefore $(34/66)^{1/2} = 0.72$ and $(34/66)^{3/2} = 0.37$.
We compare our DR4 refits with the DR3 catalogue entries of the same sources (file `dr4_prerelease_dr3`; for this
curated sample the DR3 and DR4 `source_id` are the same, which is not true in general).

In [ ]:
dr3 = load('dr4_prerelease_dr3').set_index('source_id').loc[tab.index]
ratio = pd.DataFrame({'parallax': tab.parallax_error / dr3.parallax_error, 'pmra': tab.pmra_error / dr3.pmra_error,
                      'pmdec': tab.pmdec_error / dr3.pmdec_error, 'G': tab.G, 'name': tab.name})
pull = (tab.parallax - dr3.parallax) / np.hypot(tab.parallax_error, dr3.parallax_error)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for c, m in [('parallax', 'o'), ('pmra', 's'), ('pmdec', '^')]:
    ax[0].semilogy(ratio.G, ratio[c], m, mfc='none', label=c)
ax[0].axhline((34 / 66)**0.5, color='C0', ls='--', lw=1, label='$(34/66)^{1/2}$')
ax[0].axhline((34 / 66)**1.5, color='C1', ls='--', lw=1, label='$(34/66)^{3/2}$')
for sid in tab.index[tab.name.isin(BINARIES)]:
    ax[0].annotate(NAMES[sid], (ratio.G[sid], ratio.parallax[sid]), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax[0].set(xlabel='$G$ [mag]', ylabel='uncertainty DR4 refit / DR3 catalogue'); ax[0].legend(fontsize=8, ncol=2)
ax[1].plot(tab.G, pull, 'o')
for sid in tab.index[tab.name.isin(BINARIES)]:
    ax[1].annotate(NAMES[sid], (tab.G[sid], pull[sid]), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax[1].axhspan(-2, 2, color='0.9')
ax[1].set(xlabel='$G$ [mag]', ylabel='$(\\varpi_{\\rm DR4} - \\varpi_{\\rm DR3}) / \\sigma$')
plt.tight_layout(); plt.show()
ok = ~tab.name.isin(BINARIES)
print('median ratio, 9 sources without companions: parallax {:.2f}, proper motion {:.2f}'.format(
      ratio.parallax[ok].median(), np.median(np.r_[ratio.pmra[ok], ratio.pmdec[ok]])))
print('DR4 - DR3 parallax [sigma]:', pull.round(1).to_dict())
# precision: the median improvement of the 9 sources without companions matches the T^-1/2 and T^-3/2 scalings
# within 0.1, and their parallaxes agree with DR3 within 2.5 sigma
assert abs(ratio.parallax[ok].median() - 0.72) < 0.1 and abs(np.median(np.r_[ratio.pmra[ok], ratio.pmdec[ok]]) - 0.37) < 0.1
assert (pull[ok].abs() < 2.5).all()

**Figure 5.3.** Left: uncertainty of the DR4-like refit divided by that of the DR3 catalogue, for the parallax
(circles) and the two proper-motion components (squares, triangles), against $G$; dashed: the scalings with the time
span. Right: difference between the DR4 refit and DR3 parallaxes, in units of their combined uncertainty; grey band:
$\pm 2$.

For the nine sources without companions, the parallax uncertainty drops to 0.70 of DR3 (median) and the proper-motion
uncertainty to 0.29 — both close to the scalings. (DR4 also has a better calibration than DR3, so the agreement is
partly coincidental.) The two binaries with a small or moderate orbit improve as well, but
their DR3 and DR4 parallaxes differ: HD 114762 by $-3.7\sigma$ and Gaia-4 by $+3.4\sigma$, because an unmodelled
orbit shifts the five-parameter solution by an amount that depends on which part of the orbit was observed. Gaia BH3
is the extreme case: its DR4 single-star uncertainties are *seven times worse* than in DR3, as $\epsilon = 6.6$ mas
inflates every weight. Its single-star DR4 astrometry is useless; its orbital solution (Section 5.5) is not.

## 5.4 Ten CCDs are not ten epochs

A transit takes about 40 s to cross the nine AF CCDs; a binary does not move in that time, but the attitude and
calibration errors of that transit are common to all of them. We split the $\chi^2$ of the single-star fit
(against $\sigma^2$) into a part *within* transits — residuals about each transit's weighted mean — and a part
*between* transits — the transit means about the model, each with its uncertainty
$(\sum 1/\sigma_i^2)^{-1/2}$. If the CCDs were independent and the model complete, both would be $\approx 1$ per
degree of freedom.

In [ ]:
def within_between(s, p, orbit=None):
    """chi^2/nu of the residuals of model p about the transit means (within), and of the transit means (between)."""
    r = s.centroid_pos_al.to_numpy() - design(s, orbit) @ p
    d = pd.DataFrame(dict(transit=s.transit_id.to_numpy(), w=1 / s.centroid_pos_error_al.to_numpy()**2))
    d['wr'] = d.w * r
    g = d.groupby('transit')[['w', 'wr']].sum()
    mean = g.wr / g.w                                                      # weighted mean residual per transit
    within = np.sum(d.w * (r - d.transit.map(mean))**2) / (len(d) - len(g))
    between = np.sum(g.w * mean**2) / (len(g) - len(p))
    return within, between


wb = pd.DataFrame([(NAMES[sid], *within_between(s, fit(s)[0])) for sid, s in ep.groupby('source_id')],
                  index=tab.index, columns=['name', 'within', 'between']).sort_values('between')
x = np.arange(len(wb))
plt.figure(figsize=(9, 4.5))
plt.semilogy(x - 0.1, wb.within, 'o', label='within transits')
plt.semilogy(x + 0.1, wb.between, 's', label='between transits (transit means)')
plt.axhline(1, color='k', lw=0.5)
plt.xticks(x, wb.name, rotation=45, ha='right'); plt.ylabel('$\\chi^2/\\nu$ of the single-star fit'); plt.legend(fontsize=8)
plt.show()
print(wb.round(2).to_string())
ok = ~wb.name.isin(BINARIES)
# the CCD uncertainties are right within a transit, for sources without companions; the binaries' excess is between
# transits (Gaia-4: 20 times), as an orbit must be
assert wb.within[ok].between(0.85, 1.2).all() and (wb.between[~ok] > 10 * wb.within[~ok]).all()

**Figure 5.4.** $\chi^2$ per degree of freedom of the single-star fit, split into the scatter of the CCD measurements
about their transit mean (circles) and the scatter of the transit means about the model (squares), for each source.

Within transits, the sources without companions sit at 0.95–1.10: the CCD uncertainties are right. Between transits
they range from 0.8 to 1.4, mean 1.12; with 54–89 transits per source the expected scatter of $\chi^2/\nu$ is about
$\pm 0.17$, so this is a marginal excess, consistent with a small error (tens of µas) common to the CCDs of a transit.
For the binaries the excess is between transits, by factors of 15 (Gaia-4) to $10^4$ (Gaia BH3) over the within
part: the orbit moves the star from one transit to the next. Gaia BH3 and HD 114762 also show an excess within
transits (3.8 and 9.9), which an orbit cannot cause; we do not know its origin for BH3, and for HD 114762, at
$G = 7.2$ the brightest source here, calibrating such bright images is known to be hard.

Two consequences for the rest of the chapter. We fit the CCD measurements, but the information on an orbit lives in
the transit means: there are ~60–90 independent epochs per source, not ~500–800. And the formal uncertainties of an
orbit fit are too small; we estimate them instead by **refitting bootstrap resamples of whole transits**.

## 5.5 Fitting the orbit of Gaia BH3

For fixed period $P$, eccentricity $e$ and time of periastron $T_p$, the orbital displacement is linear in the
Thiele–Innes constants $A, B, F, G$ (Chapter 1), so the full model — six source parameters plus four orbital
constants — is again solved by linear least squares. Only $(P, e, T_p)$ need a non-linear search: we scan a grid in
the three, keep the best $\chi^2$, and refine it with the Nelder–Mead simplex method. Keeping the minimum of $\chi^2$
over $e$ and $T_p$ at each $P$ gives a **periodogram**, $\chi^2_{\min}(P)$.

We weight by the CCD uncertainties alone ($\epsilon = 0$): the DR4 excess noise of this source is its unmodelled
orbit, and the orbit is now in the model.

In [ ]:
def chi2(s, orbit):
    """chi^2 against the CCD uncertainties of the source-plus-orbit model, minimised over the 10 linear parameters."""
    return fit(s, orbit, excess=False)[3]


def periodogram(s, P_grid, n_e=10, n_tp=24):
    """Minimum chi^2 over a grid in (e, tp) at each period, and the best (chi^2, P, e, tp) overall."""
    best, prof = None, []
    for P in P_grid:
        c = min((chi2(s, (P, e, tp)), P, e, tp) for e in np.linspace(0.05, 0.9, n_e)
                for tp in np.linspace(0, P, n_tp, endpoint=False))
        prof.append(c[0])
        best = c if best is None or c[0] < best[0] else best
    return np.array(prof), best


def refine(s, start):
    """Nelder-Mead on (P, e, tp), starting from start; chi^2 is minimised over the linear parameters inside."""
    f = lambda q: chi2(s, (q[0], np.clip(q[1], 0, 0.99), q[2]))
    return minimize(f, start, method='Nelder-Mead', options=dict(xatol=1e-9, fatol=1e-8, maxiter=6000)).x


def campbell(A, B, F, G):
    """Campbell elements (a0, i [deg], omega [deg], Omega [deg]) from Thiele-Innes constants, with 0 <= Omega < 180."""
    u, v = (A**2 + B**2 + F**2 + G**2) / 2, A * G - B * F
    a0 = np.sqrt(u + np.sqrt((u + v) * (u - v)))
    i = np.degrees(np.arccos(v / a0**2))
    s, d = np.arctan2(B - F, A + G), np.arctan2(-B - F, A - G)              # omega + Omega, omega - Omega
    om, Om = np.degrees((s + d) / 2), np.degrees((s - d) / 2)
    flip = (Om % 360) >= 180                                                # (omega, Omega) -> (omega + 180, Omega + 180)
    return a0, i, (om + 180 * flip) % 360, (Om + 180 * flip) % 360


def elements(s, q):
    """Orbit (P [d], e, Tp [JD], a0 [mas], i, omega, Omega [deg]) and parallax [mas] of the best fit at (P, e, tp) = q."""
    p = fit(s, tuple(q), excess=False)[0]
    A, F, B, G = p[6:10]
    a0, inc, om, Om = campbell(A, B, F, G)
    return dict(P=q[0] * 365.25, e=q[1], Tp=q[2] * 365.25 + JD_REF, a0=a0, i=inc, omega=om, Omega=Om, parallax=p[2])


def bootstrap(s, q, n=100, seed=1):
    """Elements refitted to n resamples of whole transits (drawn with replacement), each starting from q."""
    rng = np.random.default_rng(seed)
    groups = [g for _, g in s.groupby('transit_id')]
    out = []
    for _ in range(n):
        sb = pd.concat([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        out.append(elements(sb, refine(sb, q)))
    return pd.DataFrame(out)


def spread(boot, el):
    """Bootstrap uncertainty of each element: half the 16-84 percentile range of the deviations from the best fit el
    (robust to the occasional resample that converges elsewhere); angle deviations are wrapped, modulo 180 degrees for
    omega and Omega because of the (omega, Omega) -> (omega + 180, Omega + 180) ambiguity."""
    out = {}
    for k in boot.columns:
        d = boot[k] - el[k]
        if k in ('omega', 'Omega'):
            d = (d + 90) % 180 - 90
        out[k] = np.diff(np.percentile(d, [16, 84]))[0] / 2
    return pd.Series(out)


P_grid = np.geomspace(1500, 30000, 25) / 365.25              # 1500 to 30000 days, in years
prof_dr4, best = periodogram(bh, P_grid)
q_bh = refine(bh, best[1:])
p_bh, C_bh, _, chi2_bh, nu_bh = fit(bh, tuple(q_bh), excess=False)
print(f'best orbit: P = {q_bh[0] * 365.25:.1f} d, e = {q_bh[1]:.4f}; UWE {np.sqrt(chi2_bh / nu_bh):.2f} '
      f'(single-star fit: {tab.uwe_sigma_only[BH3]:.1f})')
print('within / between transits after the orbit fit: {:.2f} / {:.2f}'.format(*within_between(bh, p_bh, tuple(q_bh))))
boot_bh = bootstrap(bh, q_bh)

The orbit reduces the UWE from 82 to 1.7. What remains is the within-transit excess of Section 5.4 (3.0 in
$\chi^2/\nu$) and a similar excess between transits (2.9): errors at the 0.1-mas level that our model does not
describe, and that the bootstrap uncertainties include.

## 5.6 From Thiele–Innes constants to Campbell elements and a mass

The inverse of the Thiele–Innes relations (Halbwachs et al. 2023, Appendix A,
https://ui.adsabs.harvard.edu/abs/2023A&A...674A...9H): with $u = (A^2+B^2+F^2+G^2)/2$ and $v = AG - BF$,

$$ a_0 = \sqrt{u + \sqrt{(u+v)(u-v)}}, \qquad \cos i = v/a_0^2, $$
$$ \omega + \Omega = \operatorname{atan2}(B - F,\, A + G), \qquad \omega - \Omega = \operatorname{atan2}(-B - F,\, A - G). $$

Astrometry alone cannot tell which way the star moves along the line of sight, so $(\omega, \Omega)$ and
$(\omega + 180°, \Omega + 180°)$ describe the same projected orbit; by convention $0 \le \Omega < 180°$. From $a_0$, the
parallax $\varpi$ and $P$ follows the astrometric mass function (Kepler's third law with $a_0/\varpi$ in au and $P$ in
years),

$$ f_M = \left(\frac{a_0}{\varpi}\right)^3 \left(\frac{P}{\rm yr}\right)^{-2} M_\odot = \frac{M_2^3}{(M_1 + M_2)^2}
\quad\text{(dark companion)}, $$

so that with an estimate of the giant's mass $M_1$ we can solve for $M_2$.

In [ ]:
def mass_function(el):
    """Astrometric mass function [Msun] of a photocentre orbit (a0 and parallax in mas, P in days)."""
    return (el['a0'] / el['parallax'])**3 / (el['P'] / 365.25)**2


def companion_mass(fM, M1):
    """M2 [Msun] of a dark companion: root of M2^3 / (M1 + M2)^2 = f_M."""
    return brentq(lambda m: m**3 / (M1 + m)**2 - fM, 1e-4, 300)


ours = elements(bh, q_bh)
ours['fM'] = fM = mass_function(ours)
boot_bh['fM'] = mass_function(boot_bh)
sig_bh = spread(boot_bh, ours)
M1 = 0.76                                          # mass of the giant [Msun] (Panuzzo et al. 2024)
M2 = companion_mass(fM, M1)
# Panuzzo et al. (2024), Table 2, astrometry-only column: value and uncertainty
pub = dict(P=(4194.7, 112.3), e=(0.7262, 0.0056), a0=(27.07, 0.56), parallax=(1.6747, 0.0094), i=(110.659, 0.107),
           Omega=(136.200, 0.147), omega=(77.77, 0.66), Tp=(2458177.28, 0.98))
cmp = pd.DataFrame({k: dict(this_fit=ours[k], bootstrap_sigma=sig_bh[k], published=pub[k][0], sigma=pub[k][1],
                            pull=(ours[k] - pub[k][0]) / pub[k][1]) for k in pub}).T
print(cmp.round(4).to_string())
print(f'f_M = {fM:.2f} ± {sig_bh.fM:.2f} Msun (published 32.03 ± 0.64); with M1 = {M1} Msun: '
      f'M2 = {M2:.1f} Msun (published, with RVs: 32.70 ± 0.82)')
# reproducibility: every element agrees with the published astrometric solution within 3 of its uncertainties
assert (cmp.pull.abs() < 3).all()

Our fit reproduces every published element within two of its uncertainties (column `pull`). Our bootstrap
uncertainties (column `bootstrap_sigma`) are 1.2–2 times the published ones, except for $\omega$ (a third); Panuzzo
et al. fitted the same kind of data with the full DR4 calibration and outlier treatment, which we do not have. The largest difference is in the parallax ($-1.6\sigma$), and so in the
mass function, which scales as $\varpi^{-3}$: $f_M = 33.2\,M_\odot$ against $32.03 \pm 0.64$. Either way the companion
is a dark object of about $33\,M_\odot$: a main-sequence star of that mass would outshine the giant by a factor of
$10^4$.

Figure 5.5 shows the orbit. To display one-dimensional data on the sky we subtract the fitted source motion
(position, parallax, proper motion, colour term) from each observation; what is left is the orbit projected on the
scan direction, which we average per transit and compare with the model projected the same way.

In [ ]:
def orbit_al(s, p, q):
    """Along-scan orbital displacement of the model with parameters p (10) and (P, e, tp) = q at the observations s."""
    return design(s, tuple(q))[:, 6:] @ p[6:]


def transit_means(s, p, q):
    """Per transit: mean time, weighted mean of (data - source motion), and of the orbit model, and its uncertainty."""
    w = 1 / s.centroid_pos_error_al.to_numpy()**2
    d = pd.DataFrame(dict(transit=s.transit_id.to_numpy(), w=w, t=s.t.to_numpy(),
                          obs=w * (s.centroid_pos_al.to_numpy() - design(s)[:, :6] @ p[:6]), model=w * orbit_al(s, p, q)))
    g = d.groupby('transit').agg(t=('t', 'mean'), w=('w', 'sum'), obs=('obs', 'sum'), model=('model', 'sum'))
    return g.assign(obs=g.obs / g.w, model=g.model / g.w, err=1 / np.sqrt(g.w))


A, F, B, G = p_bh[6:10]
per = transit_means(bh, p_bh, q_bh)
tt = np.linspace(-3, 3, 2000)
X, Y = kepler_xy(tt, *q_bh)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(B * X + G * Y, A * X + F * Y, 'k-', lw=1)
x0, y0 = kepler_xy(np.array([per.t.min(), per.t.max()]), *q_bh)
ax[0].plot(B * x0 + G * y0, A * x0 + F * y0, 'o', color='C1', label='start and end of the DR4 data')
ax[0].plot(0, 0, 'k+', ms=12, label='centre of mass')
ax[0].set(xlabel='$\\Delta\\alpha_*$ [mas]', ylabel='$\\Delta\\delta$ [mas]', aspect='equal', title='fitted photocentre orbit')
ax[0].invert_xaxis(); ax[0].legend(fontsize=8)
ax[1].plot(2017.5 + per.t, per.obs, 'o', ms=4, label='data minus source motion (transit means)')
ax[1].plot(2017.5 + per.t, per.model, 'k.', ms=3, label='orbit model')
ax[1].set(xlabel='time [yr]', ylabel='along-scan orbital displacement [mas]'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Figure 5.5.** Left: the fitted photocentre orbit of Gaia BH3 on the sky (east to the left), $a_0 = 27$ mas, with the
start and end of the DR4 data (orange) and the centre of mass (cross). Right: along-scan positions minus the fitted
source motion, averaged per transit (blue), and the orbit model projected on each transit's scan direction (black).

The data and the model agree transit by transit, over a range of ±20 mas. The DR4 data cover the periastron passage
(early 2018) and 5.1 of the 11.5 years of the orbit (left); this is enough because a fast periastron passage fixes $e$
and $T_p$, and the curvature of the remaining arc fixes $P$.

## 5.7 Why DR3 did not find it

Gaia DR3 used data from 25 July 2014 to 28 May 2017. BH3 had RUWE 3.4 there, but DR3 published no orbit and no
acceleration solution for it. We cut the DR4 data to the DR3 window and repeat the analysis; for the acceleration
model we add $(g_{\alpha*}, g_\delta)\,t^2/2$ terms to the single-star design matrix.

In [ ]:
win = bh[(bh.t > (2456863.5 - JD_REF) / 365.25) & (bh.t < (2457901.5 - JD_REF) / 365.25)]
D = design(win)
sn, cs, t = D[:, 0], D[:, 1], win.t.to_numpy()
Dacc = np.column_stack([D, t**2 / 2 * sn, t**2 / 2 * cs])                  # 8 parameters: + acceleration
w, sig = win.centroid_pos_al.to_numpy(), win.centroid_pos_error_al.to_numpy()
uwe = {}
for label, M in [('single star', D), ('acceleration', Dacc)]:
    N, b = M.T @ (M / sig[:, None]**2), M.T @ (w / sig**2)
    N[5, 5] += 1 / COLOUR_PRIOR**2
    uwe[label] = np.sqrt(np.sum(((w - M @ np.linalg.solve(N, b)) / sig)**2) / (len(w) - M.shape[1]))
prof_dr3, best3 = periodogram(win, P_grid)
print(f'DR3 window: {len(win)} observations, {win.transit_id.nunique()} transits; UWE single star {uwe["single star"]:.1f}, '
      f'with acceleration {uwe["acceleration"]:.1f}')
plt.semilogy(P_grid * 365.25, prof_dr4 - prof_dr4.min() + 1, 'o-', label='DR4 data (66 months)')
plt.semilogy(P_grid * 365.25, prof_dr3 - prof_dr3.min() + 1, 's-', label='the same, DR3 window only (34 months)')
plt.xscale('log'); plt.xlabel('period P [days]'); plt.ylabel('$\\chi^2_{\\min}(P) - \\chi^2_{\\min} + 1$'); plt.legend()
plt.show()
# DR4 pins the period (chi^2 rises by > 1e3 a factor of 2 away from the best period); the DR3 window does not (< 25)
far = np.abs(np.log(P_grid * 365.25 / 4200)) > np.log(2)
print(f'DR3 window: max Delta chi^2 over 1500-30000 d = {(prof_dr3 - prof_dr3.min()).max():.1f}')
assert (prof_dr4[far] - prof_dr4.min()).min() > 1e3 and (prof_dr3 - prof_dr3.min()).max() < 25

**Figure 5.6.** Periodogram of Gaia BH3: the minimum of $\chi^2$ over eccentricity and periastron time at each trial
period, relative to the overall minimum (plus 1, for the logarithmic axis), for the DR4 data (blue) and for the same
data restricted to the DR3 time window (orange).

With DR4 data the minimum near 4200 d is sharp; $\chi^2$ is higher by $10^3$–$10^5$ at periods more than a factor of 2
away: the period is pinned. (The coarse grid of 25 periods is enough here; the refined fit of Section 5.5 does not
depend on it.) In the DR3 window, every period from 1500 to 30000 d fits about equally well ($\Delta\chi^2 \le 14$, and
Section 5.4 tells us to divide such numbers by about 3 for the correlated CCDs). Those 34 months fall between two
periastron passages, where the photocentre drifts slowly along a gently curved arc; a proper motion plus an
acceleration (UWE 2.4, against 8.5 without it) describes the arc almost as well as any orbit. DR3 could at best have
reported an acceleration; it reported nothing for this source. With 66 months and a periastron passage in the data,
the orbit — and a 33 $M_\odot$ black hole — follows. This is the general lesson for DR4: the doubled baseline turns
accelerations into orbits and RUWE-only candidates into accelerations, as Figure 1.4 predicted.

## 5.8 Two more binaries: HD 114762 and Gaia-4

Both had DR3 orbital solutions (Chapter 3), so here DR4 refines rather than discovers.

- **HD 114762**, a $G = 7.2$ late-F subgiant, was the first substellar-companion candidate found with radial
  velocities (Latham et al. 1989, https://ui.adsabs.harvard.edu/abs/1989Natur.339...38L; minimum mass $11\,M_{\rm J}$,
  $P = 84$ d). Kiefer (2019, https://ui.adsabs.harvard.edu/abs/2019A&A...632L...9K) used its DR1 excess noise to argue
  for a nearly face-on orbit, $i = 6.2^{+1.9}_{-1.3}$ degrees, and a stellar companion; DR3 published an orbit for it
  (`OrbitalTargetedSearchValidated`).
- **Gaia-4**, a $0.644\,M_\odot$ M dwarf, has a DR3 `Orbital` solution that Stefánsson et al. (2025,
  https://ui.adsabs.harvard.edu/abs/2025AJ....169..107S) confirmed with radial velocities: a planet of
  $11.8 \pm 0.7\,M_{\rm J}$, $P = 571.3 \pm 1.4$ d, $a_0 = 0.312 \pm 0.040$ mas.

We use the same machinery, with period grids uniform in frequency, spaced by a quarter of $1/T$ ($T$ the time span of
the data) so that no peak falls between grid points.

In [ ]:
targets = {'HD 114762': (3937211745905473024, 20, 1000), 'Gaia-4': (1457486023639239296, 100, 2000)}
res = {}
for name, (sid, Pmin, Pmax) in targets.items():
    s = ep[ep.source_id == sid]
    T = s.t.max() - s.t.min()
    Pg = 1 / np.arange(365.25 / Pmax, 365.25 / Pmin, 1 / (4 * T))       # uniform in frequency [1/yr]
    prof, best = periodogram(s, Pg, n_e=6, n_tp=12)
    q = refine(s, best[1:])
    p, C, _, c2, nu = fit(s, tuple(q), excess=False)
    res[name] = dict(s=s, Pg=Pg, prof=prof, q=q, p=p, el=elements(s, q), boot=bootstrap(s, q),
                     uwe=np.sqrt(c2 / nu), dr3=dr3.loc[sid])
    print(f'{name}: {len(Pg)} trial periods; best P = {q[0] * 365.25:.2f} d; UWE {np.sqrt(c2 / nu):.2f} '
          f'(single star {tab.uwe_sigma_only[sid]:.2f})')

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
for j, (name, r) in enumerate(res.items()):
    ax[0, j].semilogy(r['Pg'] * 365.25, r['prof'] - r['prof'].min() + 1, '-')
    ax[0, j].axvline(r['dr3'].period, color='C1', ls='--', lw=1, label=f'DR3 period, {r["dr3"].period:.1f} d')
    ax[0, j].set(xscale='log', xlabel='period P [days]', ylabel='$\\chi^2_{\\min}(P) - \\chi^2_{\\min} + 1$', title=name)
    ax[0, j].legend(fontsize=8)
    per = transit_means(r['s'], r['p'], r['q'])
    phase = ((per.t - r['q'][2]) / r['q'][0]) % 1
    ax[1, j].errorbar(phase, per.obs, per.err, fmt='o', ms=3, label='data minus source motion (transit means)')
    ax[1, j].plot(phase, per.model, 'k.', ms=4, label='orbit model')
    ax[1, j].set(xlabel='orbital phase (0 = periastron)', ylabel='along-scan orbital displacement [mas]')
    ax[1, j].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Figure 5.7.** Top: periodograms of HD 114762 (left) and Gaia-4 (right), as in Figure 5.6, from the DR4 epoch
astrometry; dashed: the DR3 orbital period. Bottom: along-scan positions minus the fitted source motion, averaged per
transit, against orbital phase (with uncertainties), and the orbit model projected on each transit's scan direction.

Each periodogram has one dominant minimum, at the DR3 period. The orbit is plain in the transit means: an
along-scan amplitude of up to 2 mas for HD 114762, and of 0.2–0.3 mas for Gaia-4, ten times the per-transit
uncertainty of 0.01–0.03 mas. Because the scan angle changes from transit to transit, the projected displacement at a given
phase scatters around zero rather than tracing one curve. The table below compares the elements with DR3 and with
the literature.

In [ ]:
rows = []
for name, r in res.items():
    d, el, b = r['dr3'], r['el'], spread(r['boot'], r['el'])
    a0_dr3, i_dr3, *_ = campbell(d.a_thiele_innes, d.b_thiele_innes, d.f_thiele_innes, d.g_thiele_innes)
    for k, v3, e3 in [('P', d.period, d.period_error), ('e', d.eccentricity, d.eccentricity_error),
                      ('a0', a0_dr3, np.nan), ('i', i_dr3, np.nan), ('parallax', d.nss_parallax, d.nss_parallax_error)]:
        rows.append(dict(source=name, element=k, DR4_fit=el[k], bootstrap_sigma=b[k], DR3=v3, DR3_sigma=e3,
                         pull=(el[k] - v3) / np.hypot(b[k], e3)))
el_cmp = pd.DataFrame(rows).set_index(['source', 'element'])
print(el_cmp.round(4).to_string())
# masses: Gaia-4 with M1 = 0.644 Msun (Stefansson et al. 2025); HD 114762 for M1 between 0.8 and 1.0 Msun
g4, hd = res['Gaia-4'], res['HD 114762']
MJ = 9.546e-4                                                               # Jupiter mass [Msun]
sig_g4, sig_hd = spread(g4['boot'], g4['el']), spread(hd['boot'], hd['el'])
M_g4 = companion_mass(mass_function(g4['el']), 0.644) / MJ
m_g4 = np.array([companion_mass(mass_function(row), 0.644) for _, row in g4['boot'].iterrows()]) / MJ
sig_m_g4 = np.diff(np.percentile(m_g4, [16, 84]))[0] / 2
print(f"Gaia-4: P = {g4['el']['P']:.1f} ± {sig_g4.P:.1f} d (Stefansson: 571.3 ± 1.4), "
      f"a0 = {g4['el']['a0']:.3f} ± {sig_g4.a0:.3f} mas (0.312 ± 0.040), M2 = {M_g4:.1f} ± {sig_m_g4:.1f} MJ (11.8 ± 0.7)")
print('HD 114762: i = {:.1f} ± {:.1f} deg; M2 = {:.2f} - {:.2f} Msun for M1 = 0.8 - 1.0 Msun'.format(
      hd['el']['i'], sig_hd.i, *[companion_mass(mass_function(hd['el']), m1) for m1 in (0.8, 1.0)]))
# consistency with DR3: P and e within 3 sigma for both binaries
assert (el_cmp.xs('P', level='element').pull.abs() < 3).all() and (el_cmp.xs('e', level='element').pull.abs() < 3).all()
# Gaia-4b: a0 and the companion mass agree with the RV-confirmed values within 3 sigma, and the companion is below
# the deuterium-burning limit of 13 MJ
assert abs(g4['el']['a0'] - 0.312) < 3 * np.hypot(sig_g4.a0, 0.040)
assert abs(M_g4 - 11.8) < 3 * np.hypot(sig_m_g4, 0.7) and M_g4 + 3 * sig_m_g4 < 13

**HD 114762.** The DR4 orbit agrees with DR3 ($P = 83.83 \pm 0.07$ d against $83.74 \pm 0.12$, $e \approx 0.30$,
$a_0 = 1.85$ mas against 1.80 mas). The orbit is close to face-on, $i = 9 \pm 6°$, consistent with Kiefer's
$6.2^{+1.9}_{-1.3}$ degrees. The mass function does not need $i$: it gives a companion of $0.19$–$0.22\,M_\odot$ for
any primary mass between 0.8 and $1.0\,M_\odot$ — an M dwarf, not a planet. (For such a faint companion the photocentre orbit is the primary's
orbit, so the dark-companion formula applies.) The orbit fit leaves UWE 3.1: this bright star has the large
within-transit excess of Figure 5.4, which no orbit removes.

**Gaia-4.** The planet is recovered from the DR4 epoch astrometry alone: $a_0 = 0.257 \pm 0.014$ mas and
$M_2 = 10.9 \pm 0.6\,M_{\rm J}$, within $1.3\sigma$ and $1.0\sigma$ of Stefánsson et al., and below the
13 $M_{\rm J}$ deuterium-burning limit. Our period, $579 \pm 3$ d, differs from their $571.3 \pm 1.4$ d by $2.2\sigma$. Their period
combines DR3 astrometry with radial velocities, ours is DR4 astrometry alone; we cannot tell from these data which is
biased, and we do not assert agreement. It is a reminder that bootstrap uncertainties from ~90 transits are themselves
uncertain at the 10% level, and that the DR4 catalogue orbits should be checked against radial velocities in the same
way.

## Summary

- DR4 epoch astrometry gives, for each CCD crossing, the along-scan position, its uncertainty, the scan angle, the
  time, the parallax factor and the colour factor: everything needed to refit a star.
- The DR4 single-star solution is a six-parameter weighted least-squares fit with a colour prior and weights
  $1/(\sigma^2+\epsilon^2)$; 20 lines of NumPy reproduce `gaiasupdate` exactly. Its binarity signal is the excess noise
  $\epsilon$; against the CCD uncertainties alone the quasars and single stars have UWE 0.98–1.04.
- Against DR3, parallax uncertainties shrink to 0.70 and proper-motion uncertainties to 0.29, as the doubled time
  span predicts; the parallaxes of unmodelled binaries shift by several $\sigma$.
- The CCDs of a transit share errors: the orbit information is in ~60–90 transit means, and uncertainties should come
  from resampling transits.
- A Keplerian orbit is linear in the Thiele–Innes constants for fixed $(P, e, T_p)$. We reproduce the published orbit
  of Gaia BH3 within $2\sigma$ ($f_M \approx 33\,M_\odot$), show that the DR3 time window alone constrains only an
  acceleration, confirm the face-on stellar companion of HD 114762, and recover the planet Gaia-4b.

**Exercises.** (1) Refit Gaia BH3 with weights $1/(\sigma^2 + s^2)$ and choose the jitter $s$ that makes
$\chi^2/\nu = 1$: how do the bootstrap and formal uncertainties compare now? (2) Repeat the bootstrap of Section 5.5
resampling CCD observations instead of transits: by how much does it underestimate the uncertainties, and why?
(3) Fit the Gaia-4 orbit with the period fixed at 571.3 d: by how much does $\chi^2$ rise, and how significant is that
once you divide by the $\chi^2/\nu$ of the best fit? (4) Cut the Gaia-4 data to the DR3 window: is the planet still
detected, and with which period uncertainty (DR3: $564 \pm 11$ d)?